# Clase 015 — NumPy: ufuncs y vectorización

**Parte 0** · VanderPlas cap. 2 § 2.3.

> 🎯 Abandonar `for` sobre arrays. Ufuncs = funciones C vectorizadas elementwise — el speedup real.

> ⏱️ ~75 min

## ⚙️ Setup

In [ ]:
import numpy as np
import time, tracemalloc
rng = np.random.default_rng(42)

## 1️⃣ ¿Qué es una ufunc?

Una **universal function** es una función NumPy que opera **elementwise** sobre arrays, implementada en C y vectorizada (SIMD cuando posible).

**Unarias** (un input): `np.exp`, `np.log`, `np.sin`, `np.sqrt`, `np.abs`, `np.negative`...
**Binarias** (dos inputs): `np.add`, `np.multiply`, `np.divide`, `np.power`, `np.maximum`...

Los **operadores** (`+`, `-`, `*`, `/`, `**`, `==`, `<`...) son sintaxis dulce sobre ufuncs.

## 2️⃣ El speedup en vivo

In [ ]:
N = 1_000_000
lst = list(range(N))
arr = np.arange(N)

# Versión Python
t0 = time.perf_counter()
res_py = [x*x + 2*x + 1 for x in lst]
t1 = time.perf_counter()

# Versión vectorizada
t2 = time.perf_counter()
res_np = arr*arr + 2*arr + 1
t3 = time.perf_counter()

print(f'Python loop : {(t1-t0)*1000:.1f} ms')
print(f'NumPy vec   : {(t3-t2)*1000:.1f} ms')
print(f'speedup     : {(t1-t0)/(t3-t2):.0f}×')

## 3️⃣ Operadores como ufuncs

```python
a + b   ≡  np.add(a, b)
a * b   ≡  np.multiply(a, b)
a ** b  ≡  np.power(a, b)
a == b  ≡  np.equal(a, b)
a > b   ≡  np.greater(a, b)
-a      ≡  np.negative(a)
```

Esto significa que `arr*arr + 2*arr + 1` son 4 ufuncs encadenadas — cada una alloca un array temporal. Para ahorrar memoria, usa `out=`:

In [ ]:
# Sin out=: cada operación alloca
A = rng.random(1_000_000)
tracemalloc.start()
result = A * A + 2*A + 1
_, peak1 = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f'sin out= : peak {peak1/1024:.0f} KB')

# Con out=: in-place, sin allocs
A = rng.random(1_000_000)
tracemalloc.start()
np.multiply(A, A, out=A)
np.multiply(2, A, out=A)   # nota: el segundo factor podría ser otro array
np.add(A, 1, out=A)
_, peak2 = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f'con out= : peak {peak2/1024:.0f} KB')
print(f'ratio    : {peak1/max(peak2,1):.1f}×')

## 4️⃣ Ufuncs trigonométricas, exponenciales y logarítmicas

VanderPlas tabla 2-4:

In [ ]:
x = np.linspace(0, 2*np.pi, 5)
print('x       :', x)
print('sin(x)  :', np.sin(x))
print('cos(x)  :', np.cos(x))
print()
print('exp(x)  :', np.exp(x[:3]))
print('log(...):', np.log(np.exp(x[:3])))   # log(exp(x)) ≈ x
print('sqrt    :', np.sqrt([1, 4, 9, 16]))
print('abs     :', np.abs([-3, 5, -7]))

## 5️⃣ `np.where` — ternario vectorizado

```python
np.where(cond_array, valor_si_true, valor_si_false)
```

Util para clasificar, máscaras, sustituciones:

In [ ]:
notas = np.array([2.8, 4.5, 6.1, 3.2, 7.0, 5.5])
estado = np.where(notas >= 4, 'aprobado', 'reprobado')
for n, e in zip(notas, estado):
    print(f'{n}: {e}')

## 6️⃣ ⚠️ Trampas

**Overflow silencioso** (ya visto en clase 014). NumPy no para, sólo wrap-around.

**NaN propagación**: cualquier operación con NaN produce NaN:

```python
np.array([1, 2, np.nan, 4]).sum()    # nan
np.array([1, 2, np.nan, 4]).mean()   # nan
```

**Fix**: usa las variantes `nan*`:

```python
np.nansum(arr)     # ignora NaN
np.nanmean(arr)    # ignora NaN
np.nanmedian(arr)
```

**División por cero**: produce `inf` con warning. Para silenciar (o convertir a NaN), usa `np.errstate`:

In [ ]:
a = np.array([1, 2, np.nan, 4, 5])
print(f'sum      : {a.sum()}')           # nan
print(f'nansum   : {np.nansum(a)}')      # 12
print(f'mean     : {a.mean()}')
print(f'nanmean  : {np.nanmean(a)}')

print()
with np.errstate(divide='ignore', invalid='ignore'):
    res = np.array([1, 0, -1]) / np.array([0, 0, 0])
    print('1/0, 0/0, -1/0 :', res)   # inf, nan, -inf

## ✅ Checklist

- [ ] Sé qué es una ufunc y por qué es rápida
- [ ] Reescribo `for+append` como expresión vectorizada
- [ ] Uso `out=` para ahorrar memoria
- [ ] Conozco `np.where` para ternarios vectorizados
- [ ] Manejo NaN con `nansum`/`nanmean`

## 📝 Homework

Ver `README.md`. 3 loops reescritos con benchmark, demo `out=`, `np.where`, manejo NaN.

## 🔗 Referencias

- VanderPlas cap. 2 § 2.3
- [ufuncs reference](https://numpy.org/doc/stable/reference/ufuncs.html)

➡️ **Siguiente:** [016 — Agregaciones](../016-numpy-agregaciones/README.md)